MPL Data Preparation

In [21]:
# ----------------------- Import required libraries ---------------
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import time
import random

In [22]:
# ============================
# Fix random seeds
# ============================

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [23]:
# ----------------------- Load dataset ----------------------

# Data address to run on localhost
# data = np.loadtxt('../data/MLoGPU_data3_train.csv', delimiter=',')

# Data address to run on Google Colab
data = np.loadtxt('/content/MLoGPU_data3_train.csv', delimiter=',')

X = data[..., :-1]
y = data[..., -1]

# Fix labels (make them 0-based)
y = y - 1

print(X.shape, y.shape)

(4000, 7) (4000,)


In [24]:
# ----------------------- Train / Test split -----------------------

# Use EXACT same split settings as teammate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Verify shapes
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (3000, 7) (3000,)
Test shape: (1000, 7) (1000,)


In [25]:
# ----------------------- Data normalization -----------------------

# Compute mean and std ONLY from training data
mean_train = np.mean(X_train, axis=0)
std_train = np.std(X_train, axis=0)

# Avoid division by zero
std_train[std_train == 0] = 1

# Normalize data
X_train_norm = (X_train - mean_train) / std_train
X_test_norm = (X_test - mean_train) / std_train

# Check results
print("Train mean (approx):", np.mean(X_train_norm, axis=0))
print("Train std (approx):", np.std(X_train_norm, axis=0))

Train mean (approx): [ 2.08721929e-17  3.15895458e-16 -1.23604830e-17 -1.59344760e-16
 -2.01135405e-17  4.45569507e-17  3.59305178e-16]
Train std (approx): [1. 1. 1. 1. 1. 1. 1.]


Staaaaaaaart MPL Implementation

In [26]:
# ============================
# Select device (CPU or GPU)
# ============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [27]:
# ============================
# Convert data to PyTorch tensors
# ============================

# Features → float32 (VERY important for neural networks)
X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_norm, dtype=torch.float32)

# Labels → long (required for classification with CrossEntropyLoss)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

# Print shapes
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

X_train_tensor: torch.Size([3000, 7])
y_train_tensor: torch.Size([3000])


In [28]:
# ============================
# Define MLP model
# ============================
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        # Input layer → hidden layer
        self.fc1 = nn.Linear(7, 32)

        # Activation
        self.relu = nn.ReLU()

        # Hidden → output layer (7 classes)
        self.fc2 = nn.Linear(32, 7)

    def forward(self, x):
        x = self.fc1(x)     # Linear transformation
        x = self.relu(x)    # Non-linearity
        x = self.fc2(x)     # Output layer
        return x

In [29]:
# ============================
# CPU RUN
# ============================

device_cpu = torch.device("cpu")

# Move data to CPU
X_train_cpu = X_train_tensor.to(device_cpu)
X_test_cpu  = X_test_tensor.to(device_cpu)
y_train_cpu = y_train_tensor.to(device_cpu)
y_test_cpu  = y_test_tensor.to(device_cpu)

# Create fresh CPU model
model_cpu = MLP().to(device_cpu)

criterion = nn.CrossEntropyLoss()
optimizer_cpu = torch.optim.Adam(model_cpu.parameters(), lr=0.001)

In [30]:
# ============================
# Training loop
# ============================

# Number of epochs (you can increase later)
num_epochs = 50

# Store loss values (optional, useful for plotting)
loss_history = []

for epoch in range(num_epochs):

    # Set model to training mode
    model_cpu.train()

    # Forward pass: compute predictions
    outputs = model_cpu(X_train_cpu)

    # Compute loss
    loss = criterion(outputs, y_train_cpu)

    # Backward pass: compute gradients
    optimizer_cpu.zero_grad()   # Clear old gradients
    loss.backward()         # Compute new gradients

    # Update weights
    optimizer_cpu.step()

    # Save loss
    loss_history.append(loss.item())

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/50], Loss: 1.9899
Epoch [20/50], Loss: 1.8980
Epoch [30/50], Loss: 1.8144
Epoch [40/50], Loss: 1.7374
Epoch [50/50], Loss: 1.6657


In [31]:
# ============================
# Evaluation (accuracy)
# ============================

# Set model to evaluation mode
model_cpu.eval()

# Disable gradient computation (faster)
with torch.no_grad():

    # Forward pass
    outputs = model_cpu(X_test_cpu)

    # Get predicted class (max score)
    _, predicted = torch.max(outputs, dim=1)

    # Compute accuracy
    correct = (predicted == y_test_cpu).sum().item()

    total = y_test_cpu.size(0)

    accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 44.00%


In [32]:
# ============================
# Measure training time (CPU)
# ============================
start_time = time.time()

# ---- Training loop (same as before) ----
num_epochs = 50

for epoch in range(num_epochs):
    model_cpu.train()

    outputs = model_cpu(X_train_cpu)
    loss = criterion(outputs, y_train_cpu)

    optimizer_cpu.zero_grad()
    loss.backward()
    optimizer_cpu.step()

end_time = time.time()

train_time = end_time - start_time

print(f"Training time (CPU): {train_time:.4f} seconds")

Training time (CPU): 0.1046 seconds


In [33]:
# ============================
# Measure inference time (CPU)
# ============================

model_cpu.eval()

start_time = time.time()

with torch.no_grad():
    outputs = model_cpu(X_test_cpu)
    _, predicted = torch.max(outputs, dim=1)

end_time = time.time()

inference_time = end_time - start_time

print(f"Inference time (CPU): {inference_time:.6f} seconds")

Inference time (CPU): 0.002305 seconds


Start test on GPU

In [34]:
# ============================
# Check GPU availability
# ============================
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [35]:
# ============================
# Select device
# ============================

device_gpu = torch.device("cuda")

X_train_gpu = X_train_tensor.to(device_gpu)
X_test_gpu  = X_test_tensor.to(device_gpu)
y_train_gpu = y_train_tensor.to(device_gpu)
y_test_gpu  = y_test_tensor.to(device_gpu)

In [36]:
# ============================
# Create fresh model on device
# ============================

model_gpu = MLP().to(device_gpu)

criterion = nn.CrossEntropyLoss()
optimizer_gpu = torch.optim.Adam(model_gpu.parameters(), lr=0.001)

print(next(model_gpu.parameters()).device)

cuda:0


In [37]:
# ============================
# Measure training time (GPU)
# ============================

num_epochs = 50
loss_history = []

# Synchronize before timing
if device_gpu.type == "cuda":
    torch.cuda.synchronize()

start_time = time.time()

for epoch in range(num_epochs):
    model_gpu.train()

    outputs = model_gpu(X_train_gpu)
    loss = criterion(outputs, y_train_gpu)

    optimizer_gpu.zero_grad()
    loss.backward()
    optimizer_gpu.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Synchronize after training
if device_gpu.type == "cuda":
    torch.cuda.synchronize()

end_time = time.time()
train_time_gpu = end_time - start_time

print(f"Training time (GPU): {train_time_gpu:.4f} seconds")

Epoch [10/50], Loss: 1.7976
Epoch [20/50], Loss: 1.7095
Epoch [30/50], Loss: 1.6318
Epoch [40/50], Loss: 1.5629
Epoch [50/50], Loss: 1.5011
Training time (GPU): 0.0682 seconds


In [38]:
# ============================
# Measure inference time (GPU)
# ============================

model_gpu.eval()

if device_gpu.type == "cuda":
    torch.cuda.synchronize()

start_time = time.time()

with torch.no_grad():
    outputs = model_gpu(X_test_gpu)
    _, predicted = torch.max(outputs, dim=1)

if device_gpu.type == "cuda":
    torch.cuda.synchronize()

end_time = time.time()
inference_time_gpu = end_time - start_time

print(f"Inference time (GPU): {inference_time_gpu:.6f} seconds")

Inference time (GPU): 0.000998 seconds


In [39]:
# ============================
# Evaluate accuracy on GPU
# ============================

model_gpu.eval()

with torch.no_grad():
    outputs = model_gpu(X_test_gpu)
    _, predicted = torch.max(outputs, dim=1)

    correct = (predicted == y_test_gpu).sum().item()
    total = y_test_gpu.size(0)

accuracy_gpu = 100 * correct / total

print(f"Test Accuracy (GPU): {accuracy_gpu:.2f}%")

Test Accuracy (GPU): 47.10%


In [40]:
# ============================
# Compare CPU and GPU results
# ============================

print(f"CPU training time:   {train_time:.4f} s")
print(f"GPU training time:   {train_time_gpu:.4f} s")
print(f"CPU inference time:  {inference_time:.6f} s")
print(f"GPU inference time:  {inference_time_gpu:.6f} s")
print(f"CPU accuracy:        {accuracy:.2f}%")
print(f"GPU accuracy:        {accuracy_gpu:.2f}%")

CPU training time:   0.1046 s
GPU training time:   0.0682 s
CPU inference time:  0.002305 s
GPU inference time:  0.000998 s
CPU accuracy:        44.00%
GPU accuracy:        47.10%
